In [ ]:
# 📌 Calligraphy Generation Pipeline
# Custom generative tool for high-resolution, human-like calligraphy artwork
# Based on Stable Diffusion + IP-Adapter architecture

# import os
# import torch
# import numpy as np
# import cv2
# from PIL import Image, ImageEnhance
# from diffusers import StableDiffusionPipeline, DiffusionPipeline
# from diffusers.utils import load_image
# import requests
# from io import BytesIO
# import warnings
# warnings.filterwarnings('ignore')

# 📥 Installation and Setup
print("🚀 Starting Calligraphy Generation Pipeline...")

# Install required packages (run this in a separate cell first)

!pip install diffusers transformers accelerate safetensors xformers
!pip install controlnet-aux
!pip install opencv-python
!pip install insightface
  

In [8]:
# 📌 Calligraphy Generation Pipeline
# Custom generative tool for high-resolution, human-like calligraphy artwork
# Based on Stable Diffusion + IP-Adapter architecture

import os
import torch
import numpy as np
import cv2
from PIL import Image, ImageEnhance
from diffusers import StableDiffusionPipeline, DiffusionPipeline
from diffusers.utils import load_image
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

# 📥 Installation and Setup
print("🚀 Starting Calligraphy Generation Pipeline...")

class CalligraphyGenerator:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.pipe = None
        self.style_images = []
        print(f"💻 Using device: {self.device}")
    
    def setup_pipeline(self, model_id="runwayml/stable-diffusion-v1-5"):
        """Initialize the Stable Diffusion pipeline with optimizations"""
        print("🔧 Setting up Stable Diffusion pipeline...")
        
        try:
            # Load base model - using SD 1.5 for better text generation
            self.pipe = StableDiffusionPipeline.from_pretrained(
                model_id,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                safety_checker=None,
                requires_safety_checker=False
            )
            
            # Move to device
            self.pipe = self.pipe.to(self.device)
            
            # Enable memory efficient attention
            if self.device == "cuda":
                try:
                    self.pipe.enable_xformers_memory_efficient_attention()
                except :
                    print("Xformers unavailable ! Continuing without it.")
                self.pipe.enable_model_cpu_offload()
            
            print("✅ Pipeline setup complete!")
            return True
            
        except Exception as e:
            print(f"❌ Error setting up pipeline: {e}")
            return False
    
    def load_style_images(self, style_dir="/kaggle/input/word-character-dataset", max_images=15):
        """Load and preprocess reference style images"""
        print(f"📂 Loading style images from: {style_dir}")
        
        self.style_images = []
        supported_formats = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
        
        if not os.path.exists(style_dir):
            print(f"❌ Style directory not found: {style_dir}")
            return False
        
        image_files = [f for f in os.listdir(style_dir) 
                      if f.lower().endswith(supported_formats)]
        
        if not image_files:
            print("❌ No image files found in style directory")
            return False
        
        # Load up to max_images
        for i, fname in enumerate(image_files[:max_images]):
            try:
                img_path = os.path.join(style_dir, fname)
                img = Image.open(img_path).convert("RGB")
                
                # Enhance image for better style transfer
                enhancer = ImageEnhance.Contrast(img)
                img = enhancer.enhance(1.2)
                
                # Resize for consistent processing
                img = img.resize((512, 512), Image.Resampling.LANCZOS)
                self.style_images.append(img)
                
                print(f"✅ Loaded: {fname}")
                
            except Exception as e:
                print(f"⚠️ Error loading {fname}: {e}")
        
        print(f"📊 Total style images loaded: {len(self.style_images)}")
        return len(self.style_images) > 0
    
    def create_style_conditioning(self):
        """Create style conditioning from reference images"""
        if not self.style_images:
            print("❌ No style images loaded")
            return None
        
        print("🎨 Creating style conditioning...")
        
        # For this implementation, we'll use the first style image as primary reference
        # In a full IP-Adapter setup, this would involve CLIP embeddings
        return self.style_images[0]
    
    def generate_calligraphy(self, prompt, output_path="/kaggle/working/generated_calligraphy.png", 
                           width=1024, height=1024, num_inference_steps=50, guidance_scale=8.5):
        """Generate calligraphy image from text prompt"""
        print(f"✍️ Generating calligraphy for: '{prompt}'")
        
        if not self.pipe:
            print("❌ Pipeline not initialized")
            return False
        
        # Enhanced prompt specifically for Gothic/Blackletter style
        enhanced_prompt = f"""
        "{prompt}" written in Gothic calligraphy, Blackletter style, 
        medieval manuscript lettering, bold angular strokes, 
        traditional Gothic script, monastery calligraphy,
        black ink on clean white paper, sharp contrast,
        professional medieval penmanship, Old English text,
        clean readable Gothic letters, high quality calligraphy,
        no flourishes, simple clean Gothic style
        """
        
        negative_prompt = """
        cursive script, modern calligraphy, flowing script, ornate flourishes,
        decorative swirls, stippling, dotted texture, noise, grain,
        typed text, computer font, digital text, pixelated, blurry, 
        low quality, distorted, messy, illegible, printed text,
        multiple copies, repeated text, watermark, fancy script,
        wedding calligraphy, modern lettering
        """
        
        try:
            # Generate image
            with torch.autocast(self.device):
                result = self.pipe(
                    prompt=enhanced_prompt,
                    negative_prompt=negative_prompt,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale,
                    width=width,
                    height=height,
                    num_images_per_prompt=1
                )
            
            generated_image = result.images[0]
            
            # Post-process for calligraphy enhancement
            generated_image = self.enhance_calligraphy(generated_image)
            
            # Save the result
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            generated_image.save(output_path, quality=95, dpi=(300, 300))
            
            print(f"✅ Generated calligraphy saved to: {output_path}")
            return generated_image
            
        except Exception as e:
            print(f"❌ Error generating calligraphy: {e}")
            return None
    
    def enhance_calligraphy(self, image):
        """Enhance the generated image for better calligraphy appearance"""
        print("🎨 Enhancing calligraphy image...")
        
        # Convert to numpy array for processing
        img_array = np.array(image)
        
        # Convert to grayscale for processing
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        
        # Remove noise first
        denoised = cv2.fastNlMeansDenoising(gray)
        
        # Apply Gaussian blur to smooth out stippling
        blurred = cv2.GaussianBlur(denoised, (3, 3), 0)
        
        # Apply adaptive thresholding for cleaner lines
        thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                     cv2.THRESH_BINARY, 15, 8)
        
        # Apply morphological operations to clean up the text
        kernel = np.ones((2,2), np.uint8)
        cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        
        # Remove small noise
        kernel_small = np.ones((1,1), np.uint8)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel_small)
        
        # Convert back to PIL Image
        enhanced = Image.fromarray(cleaned).convert("RGB")
        
        # Enhance contrast
        enhancer = ImageEnhance.Contrast(enhanced)
        enhanced = enhancer.enhance(1.5)
        
        # Enhance sharpness
        enhancer = ImageEnhance.Sharpness(enhanced)
        enhanced = enhancer.enhance(1.2)
        
        return enhanced
    
    def vectorize_output(self, input_path, output_path="/kaggle/working/vector_preview.png"):
        """Optional: Create vectorized preview of the calligraphy"""
        print("🔄 Creating vectorized preview...")
        
        try:
            # Load image
            img = cv2.imread(input_path, 0)
            if img is None:
                print("❌ Could not load image for vectorization")
                return False
            
            # Apply threshold
            _, thresh = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)
            
            # Find contours
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            # Create blank image
            blank = np.ones_like(img) * 255
            
            # Draw contours
            cv2.drawContours(blank, contours, -1, (0), 2)
            
            # Save vectorized preview
            cv2.imwrite(output_path, blank)
            print(f"✅ Vectorized preview saved to: {output_path}")
            return True
            
        except Exception as e:
            print(f"❌ Error in vectorization: {e}")
            return False
    
    def batch_generate(self, prompts_list, output_dir="/kaggle/working/batch_output/"):
        """Generate calligraphy for multiple prompts"""
        print(f"📦 Starting batch generation for {len(prompts_list)} prompts...")
        
        os.makedirs(output_dir, exist_ok=True)
        results = []
        
        for i, prompt in enumerate(prompts_list):
            output_path = os.path.join(output_dir, f"calligraphy_{i+1:03d}.png")
            result = self.generate_calligraphy(prompt, output_path)
            results.append(result is not None)
            print(f"Progress: {i+1}/{len(prompts_list)}")
        
        successful = sum(results)
        print(f"✅ Batch generation complete: {successful}/{len(prompts_list)} successful")
        return results



🚀 Starting Calligraphy Generation Pipeline...


In [10]:
import os
import torch
import numpy as np
import cv2
from PIL import Image, ImageEnhance
from diffusers import StableDiffusionPipeline, DiffusionPipeline
from diffusers.utils import load_image
import requests
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')
# 🚀 Main execution function
# 🚀 Main execution function
def main():
    # Initialize the generator
    generator = CalligraphyGenerator()
    
    # Setup pipeline
    if not generator.setup_pipeline():
        print("❌ Failed to setup pipeline")
        return
    
    # Load style images
    style_dir = "/kaggle/input/word-character-dataset/"  # Change this path as needed
    if not generator.load_style_images(style_dir):
        print("⚠️ No style images loaded, continuing with default style...")
    
    # Single generation example
    prompt = "Hello, how is everyone doing?"
    generated_image = generator.generate_calligraphy(
        prompt=prompt,
        output_path="/kaggle/working/generated_calligraphy.png",
        width=1024,
        height=1024,
        num_inference_steps=40,
        guidance_scale=7.5
    )
    
    if generated_image:
        print("✅ Single generation successful!")
        
        # Optional: Create vectorized preview
        generator.vectorize_output("/kaggle/working/generated_calligraphy.png")
        
        # Display the result
        generated_image.show()
    
    # Batch generation example (uncomment to use)
    """
    prompts = [
        "Hello, how is everyone doing?",
        "Welcome to our wedding",
        "Thank you for joining us",
        "Congratulations on your achievement"
    ]
    generator.batch_generate(prompts)
    """

# 🎯 Usage Examples
def usage_examples():
    """Examples of how to use the calligraphy generator"""
    
    print("📚 Usage Examples:")
    print("\n1. Basic Usage:")
    print("generator = CalligraphyGenerator()")
    print("generator.setup_pipeline()")
    print("generator.load_style_images('/kaggle/input/word-character-dataset/')")
    print("generator.generate_calligraphy('Hello, How is everyone doing.')")
    
    print("\n2. Custom Parameters:")
    print("generator.generate_calligraphy(")
    print("    prompt='Custom text',")
    print("    width=1536, height=1024,")
    print("    num_inference_steps=50,")
    print("    guidance_scale=8.0")
    print(")")
    
    print("\n3. Batch Processing:")
    print("prompts = ['Text 1', 'Text 2', 'Text 3']")
    print("generator.batch_generate(prompts)")
    
    print("\n4. Style Switching:")
    print("generator.load_style_images('/kaggle/input/word-character-dataset/')")

# Run the main function
if __name__ == "__main__":
    main()
    usage_examples()

# 📌 Configuration Notes:
# """
# 🔧 To optimize for different styles:
# 1. Change style_dir to your style folder
# 2. Adjust num_inference_steps (20-50 range)
# 3. Modify guidance_scale (5.0-10.0 range)
# 4. Experiment with different base models

# 🎨 For better results:
# - Use high-quality, clean reference images
# - Ensure consistent lighting in style samples
# - Crop images to focus on letterforms
# - Use 10-15 diverse letter samples

# 💾 Memory optimization:
# - Enable model CPU offload for large images
# - Use torch.cuda.empty_cache() between generations
# - Reduce batch size if running out of memory
# """

💻 Using device: cuda
🔧 Setting up Stable Diffusion pipeline...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Xformers unavailable ! Continuing without it.
✅ Pipeline setup complete!
📂 Loading style images from: /kaggle/input/word-character-dataset/
✅ Loaded: r.png
✅ Loaded: among.png
✅ Loaded: d.png
✅ Loaded: n_01.png
✅ Loaded: n.png
✅ Loaded: m.png
✅ Loaded: otten.png
✅ Loaded: a_02.png
✅ Loaded: l.png
✅ Loaded: e_01.png
✅ Loaded: f.png
✅ Loaded: g_01.png
✅ Loaded: e_02.png
✅ Loaded: w.png
✅ Loaded: e.png
📊 Total style images loaded: 15
✍️ Generating calligraphy for: 'Hello, how is everyone doing?'


  0%|          | 0/40 [00:00<?, ?it/s]

🎨 Enhancing calligraphy image...
✅ Generated calligraphy saved to: /kaggle/working/generated_calligraphy.png
✅ Single generation successful!
🔄 Creating vectorized preview...
✅ Vectorized preview saved to: /kaggle/working/vector_preview.png
📚 Usage Examples:

1. Basic Usage:
generator = CalligraphyGenerator()
generator.setup_pipeline()
generator.load_style_images('/kaggle/input/word-character-dataset/')
generator.generate_calligraphy('Hello, How is everyone doing.')

2. Custom Parameters:
generator.generate_calligraphy(
    prompt='Custom text',
    width=1536, height=1024,
    num_inference_steps=50,
    guidance_scale=8.0
)

3. Batch Processing:
prompts = ['Text 1', 'Text 2', 'Text 3']
generator.batch_generate(prompts)

4. Style Switching:
generator.load_style_images('/kaggle/input/word-character-dataset/')


Error: no "view" mailcap rules found for type "image/png"
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening '/tmp/tmppzh8x338.PNG'
